<a href="https://colab.research.google.com/github/KravitzLab/Murrell2026/blob/main/Figures/Murrell_2026_Fig5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Murrell 2026 Figure 5

Authors: Chantelle Murrell and Lex Kravitz<br>
Updated: 12-30-25  

In [ ]:
#@title Import dependencies
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns

# statsmodels is convenient for analytic power calculations
try:
    import statsmodels.api as sm
    from statsmodels.stats.power import TTestIndPower
except Exception:
    !pip -q install statsmodels
    import statsmodels.api as sm
    from statsmodels.stats.power import TTestIndPower

print("Ready.")


In [ ]:
#@title Import Bandit100metrics.csv
df = pd.read_csv(r"https://raw.githubusercontent.com/KravitzLab/Murrell2026/refs/heads/main/Data/SummaryStats/Bandit100_metrics.csv")

required = {"Sex", "Win-stay", "TotalAccuracy"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}. Found columns: {list(df.columns)}")

# Keep only M/F and non-missing Win-stay
df = df[df["Sex"].isin(["M","F"])].copy()
df["Win-stay"] = pd.to_numeric(df["Win-stay"], errors="coerce")
df = df.dropna(subset=["Win-stay"])

m = df.loc[df["Sex"]=="M", "Win-stay"].to_numpy()
f = df.loc[df["Sex"]=="F", "Win-stay"].to_numpy()

print("Rows after filtering:", len(df))
print(f"M: n={len(m)}, mean={m.mean():.3f}, sd={m.std(ddof=1):.3f}")
print(f"F: n={len(f)}, mean={f.mean():.3f}, sd={f.std(ddof=1):.3f}")


In [ ]:
#@title Figure 5A
bins = 12  # change if you want

plt.figure(figsize=(6,4))

plt.hist(m, bins=bins, alpha=0.35,
         label=f"M (n={len(m)})", color="dodgerblue")
plt.hist(f, bins=bins, alpha=0.35,
         label=f"F (n={len(f)})", color="red")

# means
mean_m = m.mean()
mean_f = f.mean()

# vertical drop lines at means
plt.axvline(mean_m, color="dodgerblue", linestyle="--", linewidth=1)
plt.axvline(mean_f, color="red", linestyle="--", linewidth=1)

plt.xlabel("Win-Stay (%)", fontsize=14)
plt.ylabel("Number of Mice", fontsize=14)
plt.xlim(50, 100)
plt.legend(frameon=False)
sns.despine()

# save as PDF
plt.tight_layout()
plt.savefig("Bandit100_win-stay_histogram_M_vs_F.pdf")

plt.show()

# Welch t-test
t_stat, p_val = stats.ttest_ind(m, f, equal_var=False)
print(f"Welch t-test: t = {t_stat:.4f}, p = {p_val:.4f}")

In [ ]:
#@title Figure 5B
from IPython.display import Image
Image(url='https://github.com/KravitzLab/Murrell2026/blob/main/Figures/Murrell_2026jpgs/Murrell_2026_Figure5_schematic.jpg?raw=true', width=300)

In [ ]:
#@title Calculate effect size (Cohen's d) + statistical power

def cohens_d(x, y):
    x = np.asarray(x); y = np.asarray(y)
    nx, ny = len(x), len(y)
    sx2 = x.var(ddof=1); sy2 = y.var(ddof=1)
    s_pooled = np.sqrt(((nx-1)*sx2 + (ny-1)*sy2) / (nx+ny-2))
    return (x.mean() - y.mean()) / s_pooled

d = cohens_d(m, f)

alpha = 0.05
analysis = TTestIndPower()

# achieved power given current sample sizes (approx)
power_achieved = analysis.power(effect_size=abs(d), nobs1=len(m),
                                ratio=len(f)/len(m), alpha=alpha, alternative='two-sided')

# analytic power curve for equal-N designs from 4..60
N = np.arange(4, 61)
power_curve = analysis.power(effect_size=abs(d), nobs1=N, ratio=1.0,
                             alpha=alpha, alternative='two-sided')

plt.figure(figsize=(6,4))
plt.plot(N, power_curve,color="darkorange")
plt.axhline(0.8, linestyle='--', linewidth=1, color="gray")
plt.xlabel("N per group (equal sizes)")
plt.ylabel("Power (approx)")
sns.despine()
plt.show()

# N needed for ~80% power (approx, equal groups)
n_80 = analysis.solve_power(effect_size=abs(d), power=0.8, alpha=alpha, ratio=1.0, alternative='two-sided')


print(f"Cohen's d (M - F): {d:.4f}")
print(f"Approx achieved power (alpha={alpha}, two-sided): {power_achieved:.4f}")
print(f"Approx N per group for 80% power (equal groups): {n_80:.2f}")


In [ ]:
#@title Empirical power simulation sampling 1000x at each N
rng = np.random.default_rng(0)

alpha = 0.05
Ns = np.arange(4, 61)     # N per group
n_sims = 1000             # increase to 5000+ if you want smoother estimates

def welch_p(x, y):
    return stats.ttest_ind(x, y, equal_var=False).pvalue

emp_power = np.zeros_like(Ns, dtype=float)

for i, n in enumerate(Ns):
    # sample WITHOUT replacement (valid up to min group size)
    if n > len(m) or n > len(f):
        emp_power[i] = np.nan
        continue

    pvals = np.empty(n_sims, dtype=float)
    for s in range(n_sims):
        xm = rng.choice(m, size=n, replace=False)
        xf = rng.choice(f, size=n, replace=False)
        pvals[s] = welch_p(xm, xf)
    emp_power[i] = (pvals < alpha).mean()

# Print a few key points
for target in [4, 8, 12, 20, 30, 40, 50, 60]:
    idx = np.where(Ns == target)[0]
    if len(idx):
        print(f"N={target:2d} per group: empirical power ~ {emp_power[idx[0]]:.3f}")


In [ ]:
#@title Figure 5C

analysis = TTestIndPower()
analytic = analysis.power(effect_size=abs(d), nobs1=Ns, ratio=1.0,
                          alpha=0.05, alternative='two-sided')

# --- find N where power first crosses 0.8 ---
def first_crossing(x, y, thresh=0.8):
    idx = np.where(y >= thresh)[0]
    return x[idx[0]] if len(idx) > 0 else np.nan

n_analytic = first_crossing(Ns, analytic, 0.8)
n_empirical = first_crossing(Ns, emp_power, 0.8)

# Calculate SEM for empirical power (standard error of a proportion)
# n_sims is available from the previous cell's execution
sem_empirical = np.sqrt(emp_power * (1 - emp_power) / n_sims)
# Handle potential nans introduced by emp_power being nan
sem_empirical[np.isnan(emp_power)] = np.nan

plt.figure(figsize=(6,4))
plt.plot(Ns, analytic, label="Analytic power", color="darkorange")
plt.plot(Ns, emp_power, label="Empirical power", color="blueviolet")

# Add error ribbon for empirical power (SEM)
plt.fill_between(Ns, emp_power - sem_empirical, emp_power + sem_empirical,
                 color='violet', alpha=0.2)

# horizontal 80% line
plt.axhline(0.8, linestyle="--", linewidth=1, color="black")

# vertical drop lines
plt.axvline(n_analytic, color="darkorange", linestyle=":", linewidth=1)
plt.axvline(n_empirical, color="blueviolet", linestyle=":", linewidth=1)


plt.xlabel("N per group")
plt.ylabel("Power")
plt.legend(frameon=False)
sns.despine()
# save as PDF (do this before show)
plt.tight_layout()
plt.savefig("analytic_vs_empirical_power.pdf")
plt.show()

# Below is additional information not shown in Figure 5

In [ ]:
#@title Histogram for Accuracy of Males vs. Females on Bandit100 task
# Rebuild the male/female arrays using TotalAccuracy for this second section
metric = "TotalAccuracy"

df_acc = df.copy()
df_acc[metric] = pd.to_numeric(df_acc[metric], errors="coerce")
df_acc = df_acc.dropna(subset=[metric])

m = df_acc.loc[df_acc["Sex"]=="M", metric].to_numpy()
f = df_acc.loc[df_acc["Sex"]=="F", metric].to_numpy()

print(f"Using metric: {metric}")
print(f"M: n={len(m)}, mean={m.mean():.3f}, sd={m.std(ddof=1):.3f}")
print(f"F: n={len(f)}, mean={f.mean():.3f}, sd={f.std(ddof=1):.3f}")

bins = 20  # change if you want

plt.figure(figsize=(6,4))

plt.hist(m, bins=bins, alpha=0.35,
         label=f"M (n={len(m)})", color="dodgerblue")
plt.hist(f, bins=bins, alpha=0.35,
         label=f"F (n={len(f)})", color="red")

# means
mean_m = m.mean()
mean_f = f.mean()

# vertical drop lines at means
plt.axvline(mean_m, color="dodgerblue", linestyle="--", linewidth=1)
plt.axvline(mean_f, color="red", linestyle="--", linewidth=1)

plt.xlabel("Total Accuracy (%)")
plt.ylabel("Number of Mice")
plt.xlim(50, 100)
plt.legend(frameon=False)
sns.despine()

# save as PDF (do this before show)
plt.tight_layout()
plt.savefig("Bandit100_accuracy_histogram_M_vs_F.pdf")

plt.show()

# Welch t-test
t_stat, p_val = stats.ttest_ind(m, f, equal_var=False)
print(f"Welch t-test: t = {t_stat:.4f}, p = {p_val:.4f}")



In [ ]:
#@title Calculate effect size (Cohen's d) + statistical power

def cohens_d(x, y):
    x = np.asarray(x); y = np.asarray(y)
    nx, ny = len(x), len(y)
    sx2 = x.var(ddof=1); sy2 = y.var(ddof=1)
    s_pooled = np.sqrt(((nx-1)*sx2 + (ny-1)*sy2) / (nx+ny-2))
    return (x.mean() - y.mean()) / s_pooled

d = cohens_d(m, f)

alpha = 0.05
analysis = TTestIndPower()

# achieved power given current sample sizes (approx)
power_achieved = analysis.power(effect_size=abs(d), nobs1=len(m),
                                ratio=len(f)/len(m), alpha=alpha, alternative='two-sided')

# analytic power curve for equal-N designs from 4..60
N = np.arange(4, 61)
power_curve = analysis.power(effect_size=abs(d), nobs1=N, ratio=1.0,
                             alpha=alpha, alternative='two-sided', )

plt.figure(figsize=(6,4))
plt.plot(N, power_curve, color="darkorange")
plt.axhline(0.8, linestyle='--', linewidth=1, color="gray")
plt.xlabel("N per group (equal sizes)")
plt.ylabel("Power (approx)")
sns.despine()
plt.show()

# N needed for ~80% power (approx, equal groups)
n_80 = analysis.solve_power(effect_size=abs(d), power=0.8, alpha=alpha, ratio=1.0, alternative='two-sided')


print(f"Cohen's d (M - F): {d:.4f}")
print(f"Approx achieved power (alpha={alpha}, two-sided): {power_achieved:.4f}")
print(f"Approx N per group for 80% power (equal groups): {n_80:.2f}")

In [ ]:
#@title Empirical power simulation sampling 1000x at each N
rng = np.random.default_rng(0)

alpha = 0.05
Ns = np.arange(4, 61)     # N per group
n_sims = 1000             # increase to 5000+ if you want smoother estimates

def welch_p(x, y):
    return stats.ttest_ind(x, y, equal_var=False).pvalue

emp_power = np.zeros_like(Ns, dtype=float)

for i, n in enumerate(Ns):
    # sample WITHOUT replacement (valid up to min group size)
    if n > len(m) or n > len(f):
        emp_power[i] = np.nan
        continue

    pvals = np.empty(n_sims, dtype=float)
    for s in range(n_sims):
        xm = rng.choice(m, size=n, replace=False)
        xf = rng.choice(f, size=n, replace=False)
        pvals[s] = welch_p(xm, xf)
    emp_power[i] = (pvals < alpha).mean()

# Print a few key points
for target in [4, 8, 12, 20, 30, 40, 50, 60]:
    idx = np.where(Ns == target)[0]
    if len(idx):
        print(f"N={target:2d} per group: empirical power ~ {emp_power[idx[0]]:.3f}")

In [ ]:
#@title Compare analytical vs empirical power plot

analysis = TTestIndPower()
analytic = analysis.power(effect_size=abs(d), nobs1=Ns, ratio=1.0,
                          alpha=0.05, alternative='two-sided')

# --- find N where power first crosses 0.8 ---
def first_crossing(x, y, thresh=0.8):
    idx = np.where(y >= thresh)[0]
    return x[idx[0]] if len(idx) > 0 else np.nan

n_analytic = first_crossing(Ns, analytic, 0.8)
n_empirical = first_crossing(Ns, emp_power, 0.8)

# Calculate SEM for empirical power (standard error of a proportion)
# n_sims is available from the previous cell's execution
sem_empirical = np.sqrt(emp_power * (1 - emp_power) / n_sims)
# Handle potential nans introduced by emp_power being nan
sem_empirical[np.isnan(emp_power)] = np.nan

plt.figure(figsize=(6,4))
plt.plot(Ns, analytic, label="Analytic power", color="darkorange")
plt.plot(Ns, emp_power, label="Empirical power", color="blueviolet")

# Add error ribbon for empirical power (SEM)
plt.fill_between(Ns, emp_power - sem_empirical, emp_power + sem_empirical,
                 color='blueviolet', alpha=0.2)

# horizontal 80% line
plt.axhline(0.8, linestyle="--", linewidth=1, color="black")

# vertical drop lines
plt.axvline(n_analytic, color="darkorange", linestyle=":", linewidth=1)
plt.axvline(n_empirical, color="blueviolet", linestyle=":", linewidth=1)


plt.xlabel("N per group")
plt.ylabel("Power")
plt.legend(frameon=False)
sns.despine()
# save as PDF (do this before show)
plt.tight_layout()
plt.savefig("analytic_vs_empirical_power.pdf")
plt.show()